# Temporal perturbations: end-to-end pipeline

Thin wrapper around `pipeline/scripts/`. Every active cell runs a script in a subprocess and skips
work whose output already exists unless its force flag is set. Sections 1-6 need GPUs, the datasets
in `pipeline/paths.yaml`, and the trained checkpoints; **section 7 (paper figures) runs as-is from the
results bundle shipped in `pipeline/results/`.**

## 0. Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display

REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "notebook" else Path.cwd().resolve()
SCRIPTS = REPO_ROOT / "pipeline" / "scripts"
sys.path.insert(0, str(REPO_ROOT))
from temporal_perturbations import paths

RESULTS = paths.results_root()
FIGURES = paths.path("figures_root")


def run(script, *args, output=None, force=False):
    """Run `pipeline/scripts/<script>` in a subprocess unless `output` already exists."""
    if output is not None and Path(output).exists() and not force:
        print(f"[cached] {script} -> {Path(output).relative_to(REPO_ROOT) if Path(output).is_relative_to(REPO_ROOT) else output}")
        return
    cmd = [sys.executable, str(SCRIPTS / script), *map(str, args)]
    print("$", " ".join(cmd[1:]))
    subprocess.run(cmd, check=True, cwd=REPO_ROOT)

## CONFIG

In [ ]:
# One switch re-runs everything; the per-section flags inherit from it.
DO_RERUN = False

FORCE_PREPARE = DO_RERUN
FORCE_TRAIN = DO_RERUN
FORCE_EVAL = DO_RERUN
FORCE_ANALYSIS = DO_RERUN
FORCE_FIGURES = True          # figures are cheap; always redraw from the current results

SEED_SUFFIX = ""              # run-name suffix of a replicate family, if any (sets TP_SEED_SUFFIX below)

import os
os.environ["TP_SEED_SUFFIX"] = SEED_SUFFIX

## 1. Inventory

In [ ]:
rows = []
for key in ["things_images", "things_fmri_images", "nod_images", "imagenet_root", "probe_datasets",
            "runs_root", "cliphba_runs_root", "pretrained_clip"]:
    try:
        p = paths.path(key)
        rows.append(f"| `{key}` | `{p}` | {'yes' if p.exists() else 'missing'} |")
    except KeyError:
        rows.append(f"| `{key}` | not configured | - |")
display(Markdown("| paths key | location | present |\n|---|---|---|\n" + "\n".join(rows)))

subdirs = sorted(p for p in RESULTS.iterdir())
display(Markdown("| results entry | files |\n|---|---|\n" + "\n".join(
    f"| `{p.name}` | {sum(1 for _ in p.rglob('*') if _.is_file()) if p.is_dir() else 1} |" for p in subdirs)))

## 2. Reference data

Rebuilds the neural reference RDMs shipped in `pipeline/data/` and exports the VTAB transfer datasets.

In [ ]:
if FORCE_PREPARE:
    run("preprocessing/prepare_things_fmri_rdms.py")
    run("preprocessing/prepare_nod_rdms.py")
    run("downloads/export_vtab_datasets.py")

## 3. Training

**From-scratch families** (RN50, ViT-B/16, ConvNeXt-B on ImageNet; CLIP ViT-B/32 on YFCC15M): each baseline
trains once; every perturbed run forks from the baseline checkpoint before its onset epoch and is perturbed for
one epoch. The exact arguments for all runs are in `pipeline/configs/scratch_runs.yaml`.

**CLIP-HBA** (fine-tuning on SPoSE targets): one run per perturbation type, onset epoch, and seed.

In [ ]:
import shlex
import yaml

# Fill these with the LitData-indexed training shards (see the header of scratch_runs.yaml).
TRAIN_DATA = {"imagenet_train_dir": "/path/to/imagenet_train_litdata",
              "imagenet_val_dir": "/path/to/imagenet_val_litdata",
              "yfcc15m_parquet_dir": "/path/to/yfcc15m_litdata",
              "val_dataset_root": "/path/to/cifar_cache"}

scratch_runs = yaml.safe_load(open(REPO_ROOT / "pipeline" / "configs" / "scratch_runs.yaml"))["runs"]
display(Markdown(f"{len(scratch_runs)} from-scratch runs in `scratch_runs.yaml`."))
for name, spec in (scratch_runs.items() if FORCE_TRAIN else []):
    args = spec["args"].format(runs_root=paths.path("runs_root"), **TRAIN_DATA)
    cmd = ["torchrun", "--nproc_per_node", str(spec["gpus"]), str(SCRIPTS / "training" / "train_from_scratch.py"), *shlex.split(args)]
    if (paths.path("runs_root") / name).exists():
        continue
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO_ROOT)

if FORCE_TRAIN:
    run("training/run_cliphba_perturbation.py")

## 4. Evaluations

Each producer writes under `pipeline/results/` (see the README table for the full list of invocations).

In [ ]:
EVALS = [
    ("evaluation/extract_imagenet_val_curves.py", [], "imagenet_val_curves.json"),
    ("evaluation/extract_lr_schedule.py", [], "lr_schedule.json"),
    ("evaluation/compute_spose_chance_floor.py", [], "spose_chance_floor.json"),
    ("evaluation/run_behavioral_rsa.py", [], "posthoc_rdm1854"),
    ("evaluation/run_behavioral_rsa_bn_recalibrated.py", [], "posthoc_rdm1854"),
    ("evaluation/run_subepoch_imagenet_val.py", [], "subepoch_imagenet_val"),
    ("evaluation/run_bn_recalibrated_val.py", [], "bn_recalibrated_val"),
    ("evaluation/run_linear_probe_eval.py", [], "linear_probe_eval_scratch_std"),
    ("evaluation/run_linear_probe_eval_bn_recalibrated.py", [], "linear_probe_eval_scratch_bn_recal_std"),
    ("evaluation/run_clip_imagenet_probe.py", [], "imagenet_ood_eval_scratch_clip_cls"),
    ("evaluation/run_clip_imagenet_ood_probe.py", [], "imagenet_ood_eval_scratch_clip_cls"),
    ("evaluation/run_model_to_model_rsa.py", [], "model_to_model_rsa_scratch_pp-in1k_bicubic"),
    ("evaluation/run_frrsa.py", ["--baseline_step", 5, "--include_onset"], "frrsa"),
    ("evaluation/run_frrsa.py", ["--arch", "resnet50", "--recalibrate_bn"], "frrsa"),
    ("evaluation/extract_cliphba_things_embeddings.py", ["--offset", 10], "cliphba_embeddings"),
    ("evaluation/extract_imagenet_embeddings.py", [], "imagenet_embeddings"),
    ("evaluation/extract_imagenet_embeddings_bn_recalibrated.py", [], "imagenet_embeddings"),
]
for script, args, out in EVALS:
    run(script, *args, output=RESULTS / out, force=FORCE_EVAL)

## 5. Embedding maps and analyses

In [ ]:
run("visualization/trimap_imagenet.py", output=RESULTS / "trimap", force=FORCE_ANALYSIS)
run("visualization/trimap_recovery.py", output=RESULTS / "trimap", force=FORCE_ANALYSIS)
run("analysis/analyze_global_local_structure.py",
    output=RESULTS / f"global_local_structure_analysis{SEED_SUFFIX}.json", force=FORCE_ANALYSIS)
run("analysis/analyze_distance_levels.py",
    output=RESULTS / f"distance_levels_analysis{SEED_SUFFIX}.json", force=FORCE_ANALYSIS)
run("analysis/analyze_clip_uniform_collapse.py",
    output=RESULTS / f"clip_uniform_collapse{SEED_SUFFIX}" / "summary.json", force=FORCE_ANALYSIS)

## 6. Numbers quoted in the text

In [ ]:
run("analysis/results_section_numbers.py")
run("analysis/structure_robustness.py")

## 7. Paper figures and tables

Runs from `pipeline/results/` alone.

In [ ]:
# Figure 2, Figure 3, and Figure 4's scripts also write the appendix figures
# FigS_recovery_after_onset, FigS_fidelity_separability, and FigS_transfer_retention_distance.
PAPER_FIGURES = [
    ("Figure 2", "visualization/fig2_performance_alignment.py", "Fig2_performance_alignment_defs_in_caption.png"),
    ("Figure 3", "visualization/fig3_storyboard.py", "Fig3_storyboard_defs_in_caption.png"),
    ("Figure 4", "visualization/fig5_linear_probe_transfer.py", "Fig6_linear_probe_generalization_defs_in_caption.png"),
    ("Appendix: recovery trajectories", "visualization/fig4_recovery_trajectories.py", "Fig5_recovery_trajectories_defs_in_caption.png"),
    ("Appendix: CLIP-HBA SPoSE dimensions", "visualization/figS_spose_dimensions.py", "Fig4_spose_dimensions_defs_in_caption.png"),
    ("Appendix: CKA alignment", "visualization/figS_cka_behavioral_alignment.py", "FigS_cka_behavioral_alignment.png"),
    ("Appendix: baseline behavioral alignment", "visualization/figS_baseline_behavioral_alignment.py", "Fig1b_baseline_alignment.png"),
    ("Appendix: baseline neural alignment", "visualization/figS_baseline_neural_alignment.py", "Fig_neural_trajectories_by_roi_things.png"),
    ("Appendix: neural alignment by ROI", "visualization/figS_neural_alignment_by_roi.py", "Fig_neural_critical_window_by_roi_things_canonical.png"),
    ("Appendix: CLIP uniform-image collapse", "visualization/figS_clip_uniform_collapse.py", "FigS_clip_uniform_collapse.png"),
]
for label, script, png in PAPER_FIGURES:
    run(script, output=FIGURES / png, force=FORCE_FIGURES)
run("visualization/table_transfer_probes.py", output=FIGURES / "table_transfer_probes.tex", force=FORCE_FIGURES)

In [ ]:
for label, _, png in PAPER_FIGURES:
    display(Markdown(f"### {label}"))
    display(Image(filename=str(FIGURES / png)))